# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect the available RecordSets, fields, and columns in the dataset, referencing each entity by its `@id`.

In [ ]:
# List available record sets and their @id
print('Available Record Sets:')
for record_set in meta.record_sets:
    print(f"- @id: {record_set['@id']}, Name: {record_set.get('name', '')}")

# For each record set, list its fields and columns (@id)
for record_set in meta.record_sets:
    print(f"\nFields for RecordSet @id {record_set['@id']}: ")
    if 'fields' in record_set:
        for field in record_set['fields']:
            print(f"  Field @id: {field['@id']}, Name: {field.get('name', '')}")
            if 'columns' in field and field['columns']:
                for column in field['columns']:
                    print(f"    Column @id: {column['@id']}, Name: {column.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** Adjust the `record_sets_to_load` below based on the output of the previous cell. We'll assume the primary data is in the first RecordSet found.

In [ ]:
# --- Find record sets programmatically (recommended way) ---
record_sets_to_load = [record_set['@id'] for record_set in meta.record_sets]
dataframes = dict()

for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet '{record_set_id}':")
        print(df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Modify the field `@id` as necessary depending on your actual dataset structure.

In [ ]:
# --- Choose the first record set as main (you can modify as needed) ---
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Using RecordSet '@id': {main_record_set_id}")

    # Try picking a numeric field programmatically
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_field_candidates:
        # Pick the first numeric field
        numeric_field = numeric_field_candidates[0]
    else:
        print("No numeric fields found!")
        numeric_field = None

    if numeric_field:
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try picking a textual/categorical field for grouping
        group_field_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        # Exclude obviously unhelpful columns
        exclude_fields = ['@id', 'id', 'accession', 'Accession']
        for candidate in group_field_candidates:
            if candidate not in exclude_fields:
                group_field = candidate
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric field available for analysis.")
else:
    print("No dataframes loaded. Please check your RecordSet configuration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping is possible
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We've loaded and inspected the Croissant dataset and schema, referencing all entities by their `@id`.
- Main record sets and available fields were explored.
- Performed sample data extraction, filtering, normalization and basic grouping.
- Visualized numeric field distributions and their breakdown across a chosen categorical field.

***

**Next steps:** Further domain-specific analysis or machine learning can be built upon these data curation and exploratory steps.